# Native fast catalog mode with a real CIGALE backend

This notebook demonstrates the CIGALE-like cached-grid workflow in CompoSED.

The important idea is:

1. Build the expensive rest-frame CIGALE SED grid once.
2. Save it to disk.
3. Later, load the grid and fit catalog objects by redshifting/projecting the cached spectra.

This is the workflow to use when the physical model grid has not changed, but the catalog, redshifts, masks, errors, or mass prior may change.

The example is intentionally small so it can run quickly, but it uses a real `CIGALEBackend` call:

`sfhdelayed -> bc03 -> nebular -> dustatt_modified_starburst -> dl2014 -> redshifting`


In [1]:
from pathlib import Path
import os
import sys
import time
import warnings

# Make the notebook work from a fresh checkout without requiring `pip install -e .` first.
_START_DIR = Path.cwd().resolve()
ROOT = None
for _candidate in (_START_DIR, *_START_DIR.parents):
    if (_candidate / "composed").is_dir():
        ROOT = _candidate
        break
if ROOT is None:
    raise RuntimeError("Could not find the CompoSED repository root from the current working directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Keep notebook execution self-contained on machines where ~/.matplotlib is not writable.
_MPLCONFIGDIR = ROOT / "notebooks" / "outputs" / ".mplconfig"
_MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_MPLCONFIGDIR.resolve()))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

try:
    import pandas as pd
except ImportError:  # keep the notebook usable in minimal environments
    pd = None

warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API.*",
    category=UserWarning,
)

import pcigale
from pcigale.data import SimpleDatabase as CIGALEDatabase

from composed.backends.cigale import CIGALEBackend
from composed.catalog_fast import (
    build_redshift_filter_operator,
    build_restframe_spectral_grid,
    fit_catalog_with_restframe_grid,
    load_restframe_spectral_grid,
    project_rest_grid_to_photometric_grid,
    save_restframe_spectral_grid,
)
from composed.data import SEDDataset
from composed.filters import FilterSet
from composed.parameters import ParameterSpace
from composed.priors import ChoicePrior, UniformPrior


Matplotlib is building the font cache; this may take a moment.


## 1. Output paths and run switches

`REBUILD_CACHE=True` forces the expensive CIGALE forward calls to run. Set it to `False` to reuse the saved `.npz` cache.


In [2]:
OUTDIR = ROOT / "notebooks" / "outputs" / "native_fast_catalog_cached_grid_demo"
OUTDIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH = OUTDIR / "real_cigale_rest_grid.npz"
REBUILD_CACHE = True

print("Output directory:", OUTDIR)
print("Cache path:", CACHE_PATH)
print("pcigale module:", pcigale.__file__)


Output directory: /Users/gregoire/Documents/Sedfitting/CompoSED/notebooks/outputs/native_fast_catalog_cached_grid_demo
Cache path: /Users/gregoire/Documents/Sedfitting/CompoSED/notebooks/outputs/native_fast_catalog_cached_grid_demo/real_cigale_rest_grid.npz
pcigale module: /Users/gregoire/Work/cigale-v2022.0/pcigale/__init__.py


## 2. Define the CIGALE model grid

The backend receives normal CIGALE module names and module parameters.

Two parameters are **not** part of the rest-frame grid:

- `log10_mass`: applied later as an explicit multiplicative normalization;
- `z`: applied later by redshifting the cached rest-frame spectra.

This is what makes the cached grid reusable for many objects.

A small CIGALE convention to notice: BC03 has a stellar metallicity value `0.02`, while the CIGALE nebular tables use nearby gas metallicities such as `0.019`. Here `zgas` is fixed to `0.019` explicitly.


In [3]:
MODULES = (
    "sfhdelayed",
    "bc03",
    "nebular",
    "dustatt_modified_starburst",
    "dl2014",
    "redshifting",
)

MODULE_PARAMETERS = {
    "sfhdelayed": {
        "tau_main": {"values": [1000.0, 3000.0]},      # Myr
        "age_main": {"values": [1000, 3000], "dtype": "int"},  # Myr
        "tau_burst": 50.0,
        "age_burst": 20,
        "f_burst": 0.0,
        "sfr_A": 1.0,
        "normalise": True,  # CompoSED requires this for explicit mass scaling.
    },
    "bc03": {
        "imf": 1,  # Chabrier
        "metallicity": {"values": [0.008, 0.02]},
    },
    "nebular": {
        "logU": -2.0,
        "zgas": 0.019,
        "ne": 100.0,
        "f_esc": 0.0,
        "f_dust": 0.0,
        "lines_width": 300.0,  # km/s
        "emission": True,
    },
    "dustatt_modified_starburst": {
        "E_BV_lines": {"values": [0.0, 0.2]},
        "E_BV_factor": 0.44,
        "uv_bump_wavelength": 217.5,  # nm
        "uv_bump_width": 35.0,         # nm
        "uv_bump_amplitude": 0.0,
        "powerlaw_slope": 0.0,
        "Ext_law_emission_lines": 1,
        "Rv": 3.1,
        "filters": "galex.NUV&sdss.gp",
    },
    "dl2014": {
        "qpah": 2.5,
        "umin": 1.0,
        "alpha": 2.0,
        "gamma": 0.02,
    },
    "redshifting": {
        "redshift": {"name": "z", "range": [0.05, 0.8]},
    },
}

parameter_space = ParameterSpace(
    names=("log10_mass", "tau_main", "age_main", "metallicity", "E_BV_lines", "z"),
    priors={
        "log10_mass": UniformPrior(8.0, 11.0),
        "tau_main": ChoicePrior([1000.0, 3000.0]),
        "age_main": ChoicePrior([1000.0, 3000.0]),
        "metallicity": ChoicePrior([0.008, 0.02]),
        "E_BV_lines": ChoicePrior([0.0, 0.2]),
        "z": UniformPrior(0.05, 0.8),
    },
)

backend = CIGALEBackend(modules=MODULES, module_parameters=MODULE_PARAMETERS)

filters = FilterSet([
    "galex.NUV",
    "sdss.gp",
    "sdss.rp",
    "sdss.ip",
    "sdss.zp",
    "2mass.J",
])

print("Parameter order:", parameter_space.names)
print("Filters:", filters.names)


Parameter order: ('log10_mass', 'tau_main', 'age_main', 'metallicity', 'E_BV_lines', 'z')
Filters: ('galex.NUV', 'sdss.gp', 'sdss.rp', 'sdss.ip', 'sdss.zp', '2mass.J')


## 3. Build and save the rest-frame CIGALE grid

The grid below is the expensive forward-model product. It contains CIGALE rest-frame luminosity-density spectra in `W / nm`, normalized per one solar mass formed.

For this demo the grid has only 16 models:

`2 tau_main x 2 age_main x 2 metallicity x 2 E_BV_lines`

`log10_mass` and `z` are excluded from the forward calls.


In [4]:
# A dense linear wavelength grid makes the cache projection agree closely with direct CIGALE photometry.
# Units: nm, rest-frame.
wavelength_nm = np.linspace(80.0, 2500.0, 8000)

if REBUILD_CACHE or not CACHE_PATH.exists():
    t0 = time.perf_counter()
    rest_grid = build_restframe_spectral_grid(
        backend,
        parameter_space,
        wavelengths_nm=wavelength_nm,
        max_grid_size=1000,
    )
    elapsed = time.perf_counter() - t0
    rest_grid.meta.update({
        "demo": "real CIGALE cached rest-frame grid",
        "cigale_modules": MODULES,
        "filters_used_later": filters.names,
        "elapsed_seconds_to_build": elapsed,
    })
    save_restframe_spectral_grid(rest_grid, CACHE_PATH)
    print(f"Built and saved {rest_grid.samples.shape[0]} CIGALE spectra in {elapsed:.2f} s")
else:
    print("Using existing cache:", CACHE_PATH)

rest_grid = load_restframe_spectral_grid(CACHE_PATH)
print("Loaded grid shape:", rest_grid.luminosity_w_per_nm.shape)
print("Grid parameter names:", rest_grid.parameter_names)
print("Valid models:", int(rest_grid.valid.sum()), "/", rest_grid.valid.size)
print("Mass normalization:", rest_grid.mass_normalization)
print("Luminosity unit:", rest_grid.meta.get("luminosity_unit", "W/nm"))


Built and saved 16 CIGALE spectra in 0.10 s
Loaded grid shape: (16, 8000)
Grid parameter names: ('tau_main', 'age_main', 'metallicity', 'E_BV_lines')
Valid models: 16 / 16
Mass normalization: MassNormalization.PER_SOLAR_MASS
Luminosity unit: W/nm


In [5]:
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(rest_grid.luminosity_w_per_nm.shape[0]):
    label = None
    if i < 3:
        label = f"model {i}: {dict(zip(rest_grid.parameter_names, rest_grid.samples[i]))}"
    ax.plot(rest_grid.wavelength_nm, rest_grid.luminosity_w_per_nm[i], lw=0.9, alpha=0.65, label=label)
ax.set_yscale("log")
ax.set_xlabel("Rest wavelength [nm]")
ax.set_ylabel(r"$L_\lambda$ [W nm$^{-1}$ per formed $M_\odot$]")
ax.set_title("Cached rest-frame CIGALE spectra")
ax.legend(fontsize=7, loc="best")
fig.tight_layout()
plt.show()


/var/folders/y8/jsy3hf3s05scysktw741dp6m0000gn/T/ipykernel_457/3790433834.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Sanity check: cached projection versus direct CIGALE photometry

This checks the unit path.

- Direct path: ask `CIGALEBackend.predict_photometry()` for one redshifted SED.
- Cached path: use the saved rest-frame spectrum, then apply the redshift/filter operator.

Both outputs are in maggies per formed solar mass. Small differences come from the finite wavelength grid used for the cache projection.


In [6]:
check_params = {
    "tau_main": 3000.0,
    "age_main": 3000.0,
    "metallicity": 0.02,
    "E_BV_lines": 0.2,
    "z": 0.35,
}
check_row = np.asarray([check_params[name] for name in rest_grid.parameter_names], dtype=float)
check_model_index = int(np.where(np.all(np.isclose(rest_grid.samples, check_row), axis=1))[0][0])

operator = build_redshift_filter_operator(
    rest_grid.wavelength_nm,
    filters,
    redshift=check_params["z"],
    igm_model="cigale",
)
projected_grid = project_rest_grid_to_photometric_grid(
    rest_grid,
    operator,
    age_parameter="age_main",
    age_unit="Myr",
)
fast_flux = projected_grid.flux[check_model_index]
direct_flux = backend.predict_photometry(check_params, filters).flux
relative_difference = (fast_flux - direct_flux) / direct_flux

rows = []
for band, direct, fast, rel in zip(filters.names, direct_flux, fast_flux, relative_difference):
    rows.append({
        "band": band,
        "direct_CIGALE_maggies": direct,
        "cached_grid_maggies": fast,
        "relative_difference": rel,
    })

if pd is not None:
    display(pd.DataFrame(rows))
else:
    for row in rows:
        print(row)

print("max |relative difference|:", np.nanmax(np.abs(relative_difference)))


,band,direct_CIGALE_maggies,cached_grid_maggies,relative_difference
0,galex.NUV,2.531400e-19,2.530852e-19,-0.000216
1,sdss.gp,3.767088e-19,3.767439e-19,0.000093
2,sdss.rp,7.232922e-19,7.229010e-19,-0.000541
3,sdss.ip,7.984034e-19,7.984323e-19,0.000036
4,sdss.zp,1.003655e-18,1.003917e-18,0.000261
5,2mass.J,1.229664e-18,1.230026e-18,0.000295


max |relative difference|: 0.0005408075773555709


## 5. Make a tiny mock catalog from the cached grid

The mock catalog is noiseless on purpose. The assigned `sigma` is 3% so the likelihood is well-defined, but no random noise is added. This cell validates the cache plumbing: when the data are generated by one cached model, the cached-grid fit should recover that model and its mass normalization.

For real data, replace this cell with construction of `SEDDataset` objects from your catalog fluxes, errors, masks, and upper limits.


In [7]:
truths = [
    {"object_id": "mock_0", "z": 0.20, "log10_mass": 9.2, "tau_main": 3000.0, "age_main": 3000.0, "metallicity": 0.02, "E_BV_lines": 0.2},
    {"object_id": "mock_1", "z": 0.35, "log10_mass": 9.7, "tau_main": 1000.0, "age_main": 3000.0, "metallicity": 0.008, "E_BV_lines": 0.0},
    {"object_id": "mock_2", "z": 0.55, "log10_mass": 10.1, "tau_main": 3000.0, "age_main": 1000.0, "metallicity": 0.02, "E_BV_lines": 0.0},
]

def model_index_for_truth(rest_grid, truth):
    row = np.asarray([truth[name] for name in rest_grid.parameter_names], dtype=float)
    matches = np.where(np.all(np.isclose(rest_grid.samples, row), axis=1))[0]
    if matches.size != 1:
        raise RuntimeError(f"Expected exactly one grid row for truth {truth}; got {matches}")
    return int(matches[0])

datasets = []
truth_model_indices = []
for truth in truths:
    operator = build_redshift_filter_operator(
        rest_grid.wavelength_nm,
        filters,
        redshift=truth["z"],
        igm_model="cigale",
    )
    phot_grid_at_z = project_rest_grid_to_photometric_grid(
        rest_grid,
        operator,
        age_parameter="age_main",
        age_unit="Myr",
    )
    model_index = model_index_for_truth(rest_grid, truth)
    truth_model_indices.append(model_index)

    noiseless_flux = 10.0 ** truth["log10_mass"] * phot_grid_at_z.flux[model_index]
    sigma = np.maximum(0.03 * noiseless_flux, 1.0e-40)
    datasets.append(
        SEDDataset(
            band_names=phot_grid_at_z.band_names,
            flux=noiseless_flux,
            sigma=sigma,
            metadata={"truth": truth},
        )
    )

print("Number of mock catalog objects:", len(datasets))
print("Band order:", datasets[0].band_names)
print("First object flux [maggies]:", datasets[0].flux)
print("First object sigma [maggies]:", datasets[0].sigma)


Number of mock catalog objects: 3
Band order: ('galex.NUV', 'sdss.gp', 'sdss.rp', 'sdss.ip', 'sdss.zp', '2mass.J')
First object flux [maggies]: [1.35402831e-09 2.60256288e-09 4.03586581e-09 4.80268901e-09
 5.10974148e-09 6.78635239e-09]
First object sigma [maggies]: [4.06208494e-11 7.80768865e-11 1.21075974e-10 1.44080670e-10
 1.53292244e-10 2.03590572e-10]


## 6. Fit the catalog using only the loaded rest-frame grid

This is the reusable step. No CIGALE SEDs are recomputed here. The code only:

1. groups objects by redshift,
2. builds one redshift/filter matrix per redshift,
3. projects all cached rest spectra into observed maggies,
4. profiles over `log10_mass`,
5. evaluates the Gaussian catalog likelihood.


In [8]:
t0 = time.perf_counter()
fit_result = fit_catalog_with_restframe_grid(
    rest_grid,
    datasets,
    redshifts=[truth["z"] for truth in truths],
    filters=filters,
    redshift_decimals=None,
    igm_model="cigale",
    log10_mass_bounds=(8.0, 11.0),
    age_parameter="age_main",
    age_unit="Myr",
)
elapsed = time.perf_counter() - t0
print(f"Catalog fit elapsed time: {elapsed:.3f} s")


Catalog fit elapsed time: 0.007 s


In [9]:
summary_rows = []
for i, truth in enumerate(truths):
    map_index = int(fit_result.profile_map_indices[i])
    map_values = dict(zip(rest_grid.parameter_names, fit_result.profile_map_estimates[i]))
    map_log10_mass = fit_result.log10_mass_profile[i, map_index]
    row = {
        "object_id": truth["object_id"],
        "z_fixed": truth["z"],
        "true_log10_mass": truth["log10_mass"],
        "map_log10_mass": map_log10_mass,
    }
    for name in rest_grid.parameter_names:
        row[f"true_{name}"] = truth[name]
        row[f"map_{name}"] = map_values[name]
    summary_rows.append(row)

if pd is not None:
    display(pd.DataFrame(summary_rows))
else:
    for row in summary_rows:
        print(row)


,object_id,z_fixed,true_log10_mass,map_log10_mass,true_tau_main,map_tau_main,true_age_main,map_age_main,true_metallicity,map_metallicity,true_E_BV_lines,map_E_BV_lines
0,mock_0,0.20,9.2,9.2,3000.0,3000.0,3000.0,3000.0,0.020,0.020,0.2,0.2
1,mock_1,0.35,9.7,9.7,1000.0,1000.0,3000.0,3000.0,0.008,0.008,0.0,0.0
2,mock_2,0.55,10.1,10.1,3000.0,3000.0,1000.0,1000.0,0.020,0.020,0.0,0.0


In [10]:
def cigale_filter_pivot_nm(filter_name):
    with CIGALEDatabase("filters") as db:
        filt = db.get(name=filter_name)
    return float(filt.pivot)

filter_pivots_nm = np.asarray([cigale_filter_pivot_nm(name) for name in filters.names])

fig, axes = plt.subplots(1, len(datasets), figsize=(4.2 * len(datasets), 3.4), sharey=False)
if len(datasets) == 1:
    axes = [axes]

for i, (ax, dataset, truth) in enumerate(zip(axes, datasets, truths)):
    map_index = int(fit_result.profile_map_indices[i])
    map_log10_mass = fit_result.log10_mass_profile[i, map_index]

    operator = build_redshift_filter_operator(
        rest_grid.wavelength_nm,
        filters,
        redshift=truth["z"],
        igm_model="cigale",
    )
    phot_grid_at_z = project_rest_grid_to_photometric_grid(
        rest_grid,
        operator,
        age_parameter="age_main",
        age_unit="Myr",
    )
    map_flux = 10.0 ** map_log10_mass * phot_grid_at_z.flux[map_index]

    ax.errorbar(filter_pivots_nm, dataset.flux, yerr=dataset.sigma, fmt="o", label="mock data")
    ax.plot(filter_pivots_nm, map_flux, "s--", label="MAP cached-grid model")
    ax.set_title(f"{truth['object_id']}  z={truth['z']:.2f}")
    ax.set_xlabel("Observed filter pivot [nm]")
    ax.set_ylabel("Flux [maggies]")
    ax.set_yscale("log")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.tight_layout()
plt.show()


/var/folders/y8/jsy3hf3s05scysktw741dp6m0000gn/T/ipykernel_457/1034969658.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## User walkthrough

**What data enters**

- A CIGALE module chain and parameter grid.
- A filter list using native CIGALE filter names.
- Catalog objects represented as `SEDDataset(flux, sigma, mask, upper_limit)`.

**What transformations happen**

- `build_restframe_spectral_grid` enumerates finite non-mass, non-redshift parameters and calls `CIGALEBackend.predict_rest_spectrum`.
- The cached rest-frame spectra are saved as a NumPy `.npz` file.
- `fit_catalog_with_restframe_grid` loads those spectra, redshifts them for each object, integrates through filters, and evaluates the catalog likelihood.

**Where units are converted**

- CIGALE rest spectra are stored as `W / nm` on a rest-frame wavelength grid in `nm`.
- The redshift/filter operator converts luminosity density into observed `maggies`.
- The likelihood consumes observed fluxes and sigmas in the same `maggies` units.

**Where masks/cuts are applied**

- Per-object masks live in each `SEDDataset`.
- This demo uses all bands, but the same catalog fit path honors masked bands and upper limits.

**Where normalization occurs**

- CIGALE SFH modules are forced to `normalise=True`.
- The rest-frame grid is per formed solar mass.
- `log10_mass` is not part of the CIGALE grid; it is profiled analytically during the catalog likelihood step.

**What final quantity is produced**

- `fit_result.profile_logp`: per-object log posterior over the cached non-mass grid after profiling mass.
- `fit_result.profile_map_estimates`: MAP non-mass parameters per object.
- `fit_result.log10_mass_profile`: best mass normalization for each object and grid model.

**Most important functions to audit**

- `CIGALEBackend.predict_rest_spectrum`
- `build_restframe_spectral_grid`
- `save_restframe_spectral_grid` / `load_restframe_spectral_grid`
- `build_redshift_filter_operator`
- `fit_catalog_with_restframe_grid`

**Sanity checks in this notebook**

- The cached projection is compared to direct CIGALE photometry for one SED.
- The mock catalog is generated from the cached grid itself.
- The fit recovers the injected grid parameters and `log10_mass` in the noiseless limit.
